## Example 1: Gini coefficient calculation

In [ ]:
import inequality
import numpy as np

# Example 1: Gini coefficient calculation
print("Example 1: Gini coefficient calculation")

# Create sample income distributions
income_equal = np.array([100, 100, 100, 100, 100])  # Perfect equality
income_unequal = np.array([10, 20, 30, 40, 500])    # High inequality
income_moderate = np.array([80, 90, 100, 110, 120])  # Moderate inequality

print("Income distributions:")
print(f"  Equal: {income_equal}")
print(f"  Unequal: {income_unequal}")
print(f"  Moderate: {income_moderate}")

# Calculate Gini coefficients using correct API
gini_equal = inequality.gini.Gini(income_equal).g
gini_unequal = inequality.gini.Gini(income_unequal).g
gini_moderate = inequality.gini.Gini(income_moderate).g

print("\nGini coefficients (0=perfect equality, 1=maximum inequality):")
print(f"  Equal distribution: {gini_equal:.4f}")
print(f"  Unequal distribution: {gini_unequal:.4f}")
print(f"  Moderate distribution: {gini_moderate:.4f}")

# Cross-check against the mean-absolute-difference definition: sum|xi - xj| / (2 n^2 mean)
def gini_reference(x):
    x = np.asarray(x, dtype=float)
    return np.abs(x[:, None] - x[None, :]).sum() / (2 * len(x) ** 2 * x.mean())

assert gini_equal == 0.0
assert np.isclose(gini_unequal, 2 / 3) and np.isclose(gini_moderate, 0.08)
for incomes in (income_equal, income_unequal, income_moderate):
    assert np.isclose(inequality.gini.Gini(incomes).g, gini_reference(incomes))

## Example 2: Atkinson index calculation

In [ ]:
import inequality
import numpy as np

# Example 2: Atkinson index calculation
print("Example 2: Atkinson index calculation")

# Create sample wealth distributions
wealth_data = [
    np.array([1000, 2000, 3000, 4000, 5000]),      # Linear progression
    np.array([1000, 1000, 1000, 1000, 10000]),     # One outlier
    np.array([500, 1000, 1500, 2000, 2500])        # Lower amounts
]

distribution_names = ["Linear", "Outlier", "Lower amounts"]

print("Wealth distributions:")
for i, (name, data) in enumerate(zip(distribution_names, wealth_data)):
    print(f"  {name}: {data}")

# Calculate Atkinson index with different epsilon values
epsilon_values = [0.5, 1.0, 2.0]  # Different inequality aversion parameters

print("\nAtkinson indices:")
for eps in epsilon_values:
    print(f"\nEpsilon = {eps} (inequality aversion parameter):")
    for name, data in zip(distribution_names, wealth_data):
        atkinson_idx = inequality.atkinson.atkinson(data, epsilon=eps)
        print(f"  {name:>15}: {atkinson_idx:.4f}")

# Cross-check against the textbook definition: 1 - (equally distributed equivalent income) / mean
def atkinson_reference(x, eps):
    x = np.asarray(x, dtype=float)
    if eps == 1.0:
        ede = np.exp(np.log(x).mean())
    else:
        ede = np.mean(x ** (1 - eps)) ** (1 / (1 - eps))
    return 1 - ede / x.mean()

for eps in epsilon_values:
    for data in wealth_data:
        assert np.isclose(inequality.atkinson.atkinson(data, epsilon=eps), atkinson_reference(data, eps))
# Equal incomes give an index of zero
assert np.isclose(inequality.atkinson.atkinson(np.array([500.0] * 5), epsilon=1.0), 0.0)

## Example 3: Schutz coefficient calculation

In [ ]:
import inequality
import numpy as np
import pandas as pd

# Example 3: Schutz coefficient calculation
print("Example 3: Schutz coefficient calculation")

# Create different economic scenarios
scenarios = {
    "Startup salaries": np.array([45000, 50000, 55000, 60000, 150000]),
    "Corporate salaries": np.array([80000, 85000, 90000, 95000, 100000]),
    "Mixed economy": np.array([20000, 40000, 60000, 80000, 200000])
}

print("Economic scenarios:")
for name, salaries in scenarios.items():
    print(f"  {name}: {salaries}")
    print(f"    Mean: ${np.mean(salaries):,.0f}")
    print(f"    Median: ${np.median(salaries):,.0f}")

print("\nSchutz coefficients:")
schutz_results = {}
for name, salaries in scenarios.items():
    df = pd.DataFrame({'income': salaries})
    schutz_coeff = inequality.schutz.Schutz(df, 'income').coefficient
    schutz_results[name] = schutz_coeff
    print(f"  {name:>18}: {schutz_coeff:.4f}")

# Compare with Gini coefficients
print("\nComparison with Gini coefficients:")
for name, salaries in scenarios.items():
    gini_coeff = inequality.gini.Gini(salaries).g
    schutz_coeff = schutz_results[name]
    print(f"  {name}: Gini={gini_coeff:.4f}, Schutz={schutz_coeff:.4f}")

# The original Schutz coefficient sums 10 * (x / mean - 1) over the incomes above the mean,
# and the Schutz distance is half the sum of |income share - population share|
def schutz_reference(x):
    x = np.asarray(x, dtype=float)
    ratio = x / x.mean()
    return 10 * (ratio[ratio > 1] - 1).sum(), np.abs(x / x.sum() - 1 / len(x)).sum() / 2

for name, salaries in scenarios.items():
    result = inequality.schutz.Schutz(pd.DataFrame({'income': salaries}), 'income')
    coefficient, distance = schutz_reference(salaries)
    assert np.isclose(result.coefficient, coefficient), name
    assert np.isclose(result.distance, distance), name
assert np.isclose(schutz_results["Corporate salaries"], 5 / 3)

## Example 4: Comparative inequality analysis

In [ ]:
import inequality
import numpy as np
import pandas as pd

# Example 4: Comparative inequality analysis
print("Example 4: Comparative inequality analysis")

# Create datasets representing different countries/regions
regions = {
    "Region A": np.array([25000, 30000, 35000, 40000, 45000, 50000]),
    "Region B": np.array([15000, 25000, 35000, 45000, 55000, 120000]),
    "Region C": np.array([10000, 15000, 20000, 25000, 30000, 35000])
}

print("Regional income distributions:")
for name, incomes in regions.items():
    print(f"  {name}: {incomes}")

# Calculate multiple inequality measures for comparison
print("\nInequality measures comparison:")
print(f"{'Region':>10} {'Gini':>8} {'Atkinson':>12} {'Schutz':>8}")
print("-" * 40)

for name, incomes in regions.items():
    gini_val = inequality.gini.Gini(incomes).g
    atkinson_val = inequality.atkinson.atkinson(incomes, epsilon=1.0)
    df = pd.DataFrame({'income': incomes})
    schutz_val = inequality.schutz.Schutz(df, 'income').coefficient
    
    print(f"{name:>10} {gini_val:>8.4f} {atkinson_val:>12.4f} {schutz_val:>8.4f}")

# Rank regions by inequality
gini_ranking = sorted(regions.keys(), key=lambda x: inequality.gini.Gini(regions[x]).g)
print(f"\nRegions ranked by inequality (Gini, low to high):")
for i, region in enumerate(gini_ranking, 1):
    print(f"  {i}. {region}")

assert gini_ranking == ["Region A", "Region C", "Region B"]
assert np.isclose(inequality.gini.Gini(regions["Region A"]).g, 35 / 270)

## Example 5: Inequality sensitivity analysis

In [ ]:
import inequality
import numpy as np

# Example 5: Inequality sensitivity analysis
print("Example 5: Inequality sensitivity analysis")

# Base income distribution
base_incomes = np.array([20000, 30000, 40000, 50000, 60000])
print(f"Base income distribution: {base_incomes}")
print(f"Base Gini coefficient: {inequality.gini.Gini(base_incomes).g:.4f}")

# Test sensitivity to changes
modifications = [
    ("Double top earner", np.array([20000, 30000, 40000, 50000, 120000])),
    ("Add high earner", np.array([20000, 30000, 40000, 50000, 60000, 200000])),
    ("Reduce bottom earner", np.array([5000, 30000, 40000, 50000, 60000])),
    ("More equal", np.array([35000, 40000, 45000, 50000, 55000]))
]

print("\nSensitivity analysis:")
print(f"{'Scenario':>20} {'Gini':>8} {'Change':>8}")
print("-" * 40)

base_gini = inequality.gini.Gini(base_incomes).g
for description, modified_incomes in modifications:
    new_gini = inequality.gini.Gini(modified_incomes).g
    change = new_gini - base_gini
    print(f"{description:>20} {new_gini:>8.4f} {change:>+8.4f}")

# Test Atkinson sensitivity to epsilon
print(f"\nAtkinson index sensitivity to epsilon parameter:")
epsilon_values = [0.1, 0.5, 1.0, 1.5, 2.0]
for eps in epsilon_values:
    atkinson_val = inequality.atkinson.atkinson(base_incomes, epsilon=eps)
    print(f"  Epsilon = {eps:>3.1f}: {atkinson_val:.4f}")

assert np.isclose(base_gini, 0.2)
ginis = {description: inequality.gini.Gini(incomes).g for description, incomes in modifications}
# Widening the gap raises the Gini coefficient; narrowing it lowers it
assert ginis["Double top earner"] > base_gini and ginis["Add high earner"] > base_gini
assert ginis["Reduce bottom earner"] > base_gini and ginis["More equal"] < base_gini
# The Atkinson index grows with inequality aversion
atkinson_values = [inequality.atkinson.atkinson(base_incomes, epsilon=eps) for eps in epsilon_values]
assert all(a < b for a, b in zip(atkinson_values, atkinson_values[1:]))